# VMD3 RADC Fast-Time Verification

Verification of raw RADC (raw ADC / beat-frequency) data recorded from the
VMD3 61 GHz FMCW radar, RSET 1, 2D mode.

**Measurement:** 20x18 cm metal plate on a Zaber linear stage at 6 m range, moving sinusoidally at 1.6 mm displacement, 0.2 Hz motion frequency. VMD3 stationary. Single RADC-only `.bin` file.

**Goal (this notebook):** condition and inspect the raw *fast-time* chirp data as a sanity/verification step before manually computing the range FFT (RFFT) and comparing it against the VMD3's FPGA-generated RFFT.

**Outline**
1. Imports and setup
2. Import data and set up the RADC frame (per-channel)
3. Filter, time-domain I/Q plot, I/Q constellation
4. (Parked) RFFT + slow-time displacement / FFT scaffolding

## 1. Imports and Setup

In [ ]:
# %%
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# ---------------------------------------------------------------------
# File path
# ---------------------------------------------------------------------
BINARY_FILEPATH = (
    "/home/shaylon/repos/ALiSM_Python_copy/vmd3-twotarget/data/radc/2026-07-17/2026-07-17_16-31-39.bin"
)

# ---------------------------------------------------------------------
# Radar configuration (RSET 1, 2D mode) — from param.md
# ---------------------------------------------------------------------
CONFIG_MODE   = "2D"        # "2D" or "3D"
HEADER        = b'RADC'

N_SAMPLES     = 128         # samples per chirp (fast-time)
N_CHIRPS      = 64          # chirps per frame  (slow-time within a frame)
N_CHANNELS    = 4           # RX channels (2D mode)

FS_FAST       = 2e6         # fast-time ADC sample rate (Hz) — 2 MHz
MAX_RANGE     = 10.0        # m   (RSET 1 max range)
RANGE_RES     = 0.0782      # m   (RSET 1 range resolution, 7.82 cm/bin)
F_CENTER      = 61.06e9     # Hz  (RSET 1 center frequency)
LAMBDA        = 3e8 / F_CENTER   # ~4.91 mm

# Frame-rate (slow-time) parameters — kept for the parked step 4
TIME_STEP     = 0.13        # s   (frame repetition time)
FS_SLOW       = 1.0 / TIME_STEP  # ~7.7 Hz frame rate

# ---------------------------------------------------------------------
# Filter parameters
# ---------------------------------------------------------------------
# Order is confirmed: 6th-order Butterworth low-pass.
FILT_ORDER = 6

# NOTE: cutoff (FILT_FC) and sample rate (FILT_FS) are intentionally NOT
# hardcoded yet. They depend on which axis is filtered:
#   - Fast-time (chirp samples): FILT_FS = FS_FAST = 2e6 Hz. A meaningful
#     low-pass cutoff here is in the kHz range, NOT 1 Hz. A 1 Hz cutoff on a
#     2 MHz-sampled axis would erase the beat tone entirely.
#   - Slow-time (per-frame phasor): FILT_FS = FS_SLOW ~ 7.7 Hz, where a
#     sub-Hz cutoff (~1 Hz) is sensible for isolating the 0.2 Hz motion.
# Set these once the intended axis + cutoff are confirmed.
FILT_FS = FS_SLOW    # default to fast-time per interpretation (b)
FILT_FC = 1.0       # <-- TO CONFIRM: cutoff frequency in Hz

# ---------------------------------------------------------------------
# Channel selection — single channel for now (0-indexed: 0,1,2,3)
# ---------------------------------------------------------------------
CHANNEL = 3

## 2. Import Data and Set Up the RADC Frame

Carried over verbatim from `vmd3_process_bin.py` to keep the notebook self-contained and the decode conventions identical.

`decode_radc_2d` returns a complex cube of shape **(samples, chirps, channels)** = (128, 64, 4), with `I + 1j*Q` per the recorded byte layout (Q at even int16 offsets, I at odd).

In [ ]:
def get_frames(filepath, header, config_mode):
    """
    Read a .bin file and extract all valid RADC frame payloads.
      - find every occurrence of the header bytes
      - read the 4-byte little-endian length that follows
      - keep only frames whose length matches the configured mode
      - reject frames whose payload accidentally contains the header
    """
    with open(filepath, 'rb') as f:
        raw_data = f.read()

    expected_len = 131072 if config_mode == "2D" else 196608
    frames = []

    start = 0
    while True:
        idx = raw_data.find(header, start)
        if idx == -1:
            break

        if idx + 8 > len(raw_data):
            break
        payload_length = int.from_bytes(
            raw_data[idx + 4:idx + 8], byteorder='little', signed=False
        )

        if payload_length == expected_len:
            payload_start = idx + 8
            payload_end = payload_start + payload_length
            if payload_end <= len(raw_data):
                payload = raw_data[payload_start:payload_end]
                if header not in payload:
                    frames.append(payload)

        start = idx + 1

    return frames


def decode_radc_2d(frame):
    """
    Decode a 2D-mode RADC payload into a complex cube.
    Output shape: (128 samples, 64 chirps, 4 channels)

    Byte layout (per sweep of 2048 bytes, 64 sweeps total):
      - 4 channel blocks of 512 bytes each
      - Within each block: Q at offsets 0,4,8,... and I at offsets 2,6,10,...
      - Each sample is int16 little-endian
    """
    if len(frame) != 131072:
        raise ValueError(f'INVALID FRAME LENGTH FOR 2D MODE: {len(frame)}')

    raw = np.frombuffer(frame, dtype='<i2')   # 65536 int16 values
    raw = raw.reshape(64, 4, 256)             # (sweeps, channels, Q/I interleaved)

    q_values = raw[:, :, 0::2]   # Q at even positions (offsets 0, 4, 8, ...)
    i_values = raw[:, :, 1::2]   # I at odd positions  (offsets 2, 6, 10, ...)

    complex_data = i_values.astype(np.float64) + 1j * q_values.astype(np.float64)
    # shape: (sweeps=64, channels=4, samples=128)

    cube = np.transpose(complex_data, (2, 0, 1))  # -> (samples, chirps, channels)
    return cube 


# ---- Load all RADC frames from the file ----
frames = get_frames(BINARY_FILEPATH, HEADER, CONFIG_MODE)
if not frames:
    raise RuntimeError('NO FRAMES FOUND')

n_frames = len(frames)
print(f'Loaded {n_frames} frames from:\n  {BINARY_FILEPATH}')

# ---- Decode every frame into a stack of cubes ----
# cubes[f] has shape (samples, chirps, channels) = (128, 64, 4)
cubes = np.stack([decode_radc_2d(f) for f in frames], axis=0)
print(f'Decoded cube stack shape (frames, samples, chirps, channels): {cubes.shape}')

# ---- Trim startup frames (transient at the very beginning) ----
# Mirrors TRIM_FRAMES in vmd3_process_bin.py. Drops the first N frames, which
# can carry a startup transient in the first chirps.
TRIM_FRAMES = 1
if TRIM_FRAMES > 0:
    cubes = cubes[TRIM_FRAMES:]
    print(f'Trimmed first {TRIM_FRAMES} frame(s) -> {cubes.shape[0]} frames remain.')

# ---- Select a single channel for this verification pass ----
# ch_data has shape (frames, samples, chirps)
ch_data = cubes[:, :, :, CHANNEL]
print(f'Channel {CHANNEL} data shape (frames, samples, chirps): {ch_data.shape}')

### Inspection indices

`INSPECT_FRAME` / `INSPECT_CHIRP` select which frame/chirp the `'chirp'` and `'frame'` scopes (and the bottom sanity check) refer to. The actual single `chirp` array is extracted at the bottom of the notebook, just above the sanity check, since that is the only place it is consumed.

In [ ]:
INSPECT_FRAME = 5    # which frame to look at (0-indexed)
INSPECT_CHIRP = 0    # which chirp within that frame (0-indexed)

### Plot scope selector

Choose how much fast-time data feeds the time-domain and constellation plots:

- `'chirp'`   : one chirp (length 128) — frame `INSPECT_FRAME`, chirp `INSPECT_CHIRP`
- `'frame'`   : all 64 chirps of frame `INSPECT_FRAME`, concatenated (8192 samples)
- `'dataset'` : all chirps of all frames, concatenated (large)

`PLOT_AVERAGE` (optional): instead of concatenating, coherently average the chirps into one representative 128-sample chirp. This is a processing step (raises SNR, assumes the target is ~stationary over the averaged chirps), not raw data — leave it False for pure raw-data verification.

Note: concatenation makes the constellation *denser*, not *rounder*. It's still stacked fast-time beat-tone rotations, never the slow-time displacement circle.

In [ ]:
PLOT_SCOPE   = 'fixed_sample'   # 'chirp' | 'frame' | 'dataset' | 'fixed_sample'
PLOT_AVERAGE = False     # if True, average chirps instead of concatenating

# Optional subsampling for dataset scope (keep every Nth sample for readability).
# Set to 1 to disable.
DATASET_SUBSAMPLE = 1

# ---- 'fixed_sample' scope options -----------------------------------
# Fix one fast-time sample index and step through slow-time (chirps/frames).
# NOTE: a fixed fast-time sample is NOT the same as an FFT range bin. This is
# the no-FFT way to look at motion: the picked sample's phase still carries the
# target's 0.2 Hz motion, but it is a single un-integrated sample (worse SNR
# than the FFT-at-bin value) and is NOT range-gated to 6 m. Expect a noisier
# version of the slow-time sinusoid, not as clean as the RFFT path.
FAST_SAMPLE_IDX  = 78          # which fast-time sample to pick from each chirp
FIXED_SAMPLE_SUB = 'one_per_frame'  # 'all_chirps' | 'one_per_frame'
FIXED_SAMPLE_CHIRP = 31         # which chirp to keep when 'one_per_frame'


def build_iq_signal(ch_data, scope, frame_idx, chirp_idx,
                    average=False, subsample=1,
                    fast_sample_idx=78, fixed_sample_sub='all_chirps',
                    fixed_sample_chirp=0):
    """
    Assemble the complex fast-time signal to plot, per the chosen scope.

    ch_data : (frames, samples, chirps) complex, single channel
    Returns a 1-D complex array.
    """
    if scope == 'chirp':
        sig = ch_data[frame_idx, :, chirp_idx]

    elif scope == 'frame':
        # (samples, chirps) for this frame
        block = ch_data[frame_idx]                      # (128, 64)
        if average:
            sig = block.mean(axis=1)                    # (128,)
        else:
            # concatenate chirps end-to-end -> (chirps*samples,)
            sig = block.T.reshape(-1)                   # chirp0, chirp1, ...

    elif scope == 'dataset':
        # (frames, samples, chirps)
        if average:
            # average across all chirps of all frames -> (128,)
            sig = ch_data.mean(axis=(0, 2))
        else:
            # concatenate every chirp of every frame, in (frame, chirp) order
            # transpose to (frames, chirps, samples) then flatten
            sig = np.transpose(ch_data, (0, 2, 1)).reshape(-1)
            if subsample > 1:
                sig = sig[::subsample]

    elif scope == 'fixed_sample':
        # Fix the fast-time sample index, sweep slow-time.
        # ch_data[:, fast_sample_idx, :] -> (frames, chirps)
        plane = ch_data[:, fast_sample_idx, :]
        if fixed_sample_sub == 'all_chirps':
            # frame0: chirp0..63, frame1: chirp0..63, ...
            sig = plane.reshape(-1)
        elif fixed_sample_sub == 'one_per_frame':
            # one chosen chirp per frame -> (frames,)
            sig = plane[:, fixed_sample_chirp]
        else:
            raise ValueError(
                "FIXED_SAMPLE_SUB must be 'all_chirps' or 'one_per_frame'"
            )
    else:
        raise ValueError(
            "PLOT_SCOPE must be 'chirp', 'frame', 'dataset', or 'fixed_sample'"
        )

    return sig


iq_signal = build_iq_signal(
    ch_data, PLOT_SCOPE, INSPECT_FRAME, INSPECT_CHIRP,
    average=PLOT_AVERAGE, subsample=DATASET_SUBSAMPLE,
    fast_sample_idx=FAST_SAMPLE_IDX,
    fixed_sample_sub=FIXED_SAMPLE_SUB,
    fixed_sample_chirp=FIXED_SAMPLE_CHIRP,
)

if PLOT_SCOPE == 'fixed_sample':
    scope_desc = f"fixed_sample[{FAST_SAMPLE_IDX}] ({FIXED_SAMPLE_SUB})"
else:
    scope_desc = PLOT_SCOPE + (' (averaged)' if PLOT_AVERAGE else '')
print(f'Plot scope: {scope_desc}  ->  {len(iq_signal)} samples')

## 3. Filter, Time-Domain I/Q Plot, I/Q Constellation

This is the core verification view. We operate on a single **fast-time** chirp (length 128, sampled at `FS_FAST` = 2 MHz).

- Raw I/Q vs sample index and the raw I/Q constellation always render — they need no cutoff and show the raw conditioned data directly.
- The 6th-order Butterworth low-pass overlay renders only once `FILT_FC` is set, since a fast-time cutoff must be chosen against the 2 MHz rate (kHz-range), not reused from the CW slow-time 1 Hz value.

In [ ]:
def lowpass_filter(x, fc, fs, order=FILT_ORDER):
    """
    Zero-phase Butterworth low-pass. Operates on a real 1-D array.
    Returns x unchanged if it's too short for filtfilt padding.
    """
    x = np.asarray(x, dtype=float)
    nyq = fs / 2.0
    if not (0 < fc < nyq):
        raise ValueError(f"Cutoff fc={fc} must be between 0 and Nyquist={nyq}.")
    b, a = butter(order, fc / nyq, btype="low", analog=False)
    padlen = 3 * (max(len(a), len(b)) - 1)
    if len(x) <= padlen:
        return x.copy()
    return filtfilt(b, a, x, padlen=padlen)


# %%
# ---- Split the selected signal into I and Q (fast-time) ----
I_raw = np.real(iq_signal)
Q_raw = np.imag(iq_signal)
n = np.arange(len(iq_signal))   # fast-time sample index

# ---- Optional filtering (only if a cutoff has been set) ----
do_filter = FILT_FC is not None
if do_filter:
    I_filt = lowpass_filter(I_raw, FILT_FC, FILT_FS, FILT_ORDER)
    Q_filt = lowpass_filter(Q_raw, FILT_FC, FILT_FS, FILT_ORDER)
    print(f'Filtered: order {FILT_ORDER} Butterworth, '
          f'fc={FILT_FC} Hz, fs={FILT_FS:g} Hz')
else:
    print('FILT_FC is None -> showing RAW I/Q only. '
          'Set FILT_FC (against FS_FAST=2 MHz) to enable the filtered overlay.')

### Time-domain I/Q

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
fig.suptitle(f'Slow-time I/Q — scope: {scope_desc}, channel {CHANNEL}')

# axs[0].plot(n, I_raw, label='I (raw)', alpha=0.8)
if do_filter:
    axs[0].plot(n, I_filt, label='I (filtered)', linewidth=2)
axs[0].set_ylabel('I amplitude')
axs[0].legend(); axs[0].grid(True, alpha=0.3)

# axs[1].plot(n, Q_raw, label='Q (raw)', alpha=0.8, color='tab:orange')
if do_filter:
    axs[1].plot(n, Q_filt, label='Q (filtered)', linewidth=2, color='tab:orange')
axs[1].set_xlabel('Slow-time frame index')
axs[1].set_ylabel('Q amplitude')
axs[1].legend(); axs[1].grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### I/Q constellation

On fast-time this traces the in-chirp beat tone rotating in the I/Q plane, not the slow-time displacement phasor. It is a "does the I/Q look coherent and well-behaved" check, not an RCS/displacement measurement.

In [ ]:
# Auto-select line vs. scatter based on point count
LINE_POINT_LIMIT = 2000   # connected line only below this many points

plt.figure(figsize=(6, 6))
if len(iq_signal) <= LINE_POINT_LIMIT:
    # plt.plot(I_raw, Q_raw, '-', linewidth=0.8, alpha=0.6, label='raw')
    if do_filter:
        plt.plot(I_filt, Q_filt, '-', linewidth=1.5, label='filtered')
        plt.legend()
else:
    # plt.plot(I_raw, Q_raw, ',', alpha=0.3, label='raw')   # ',' = pixel marker
    if do_filter:
        plt.plot(I_filt, Q_filt, ',', alpha=0.5, label='filtered')
        plt.legend()

plt.axis('equal')
plt.xlabel('I (real)')
plt.ylabel('Q (imag)')
plt.title(f'I/Q Circle — scope: {scope_desc}, channel {CHANNEL}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Unwrap, Displacement, and Spectra

This operates on `iq_signal` as a **slow-time** phasor stream. It only makes physical sense when `iq_signal` is one value per slow-time step — i.e. `PLOT_SCOPE = 'fixed_sample'` with `FIXED_SAMPLE_SUB = 'one_per_frame'` (uniform 7.7 Hz frame rate). On fast-time scopes the unwrap/displacement below is meaningless; a guard warns if the rate looks wrong.

Deliverables:
1. Displacement vs time   (from unwrapped I/Q phase)
2. FFT of the I/Q signal   (motion spectrum, from the time-domain signal)
3. FFT of the displacement (motion spectrum, from the displacement waveform)

In [ ]:
# ---- Slow-time sample rate for this stream ----
# one_per_frame -> frame rate; otherwise this analysis is not well-posed.
if PLOT_SCOPE == 'fixed_sample' and FIXED_SAMPLE_SUB == 'one_per_frame':
    fs_slow = FS_SLOW
else:
    fs_slow = FS_SLOW
    print('[WARN] Step 4 assumes a one-value-per-slow-time-step stream. '
          "Use PLOT_SCOPE='fixed_sample', FIXED_SAMPLE_SUB='one_per_frame' "
          'for a physically meaningful displacement.')

# Optionally low-pass the phasor before demod to reduce single-sample noise.
# Uses the slow-time rate, NOT FS_FAST.
PHASOR_FILTER_FC = None    # e.g. 1.0 (Hz) to isolate the ~0.2 Hz motion; None = off

iq_demod = iq_signal.copy()
if PHASOR_FILTER_FC is not None:
    I_f = lowpass_filter(np.real(iq_demod), PHASOR_FILTER_FC, fs_slow, FILT_ORDER)
    Q_f = lowpass_filter(np.imag(iq_demod), PHASOR_FILTER_FC, fs_slow, FILT_ORDER)
    iq_demod = I_f + 1j * Q_f
    print(f'Phasor low-pass: order {FILT_ORDER}, fc={PHASOR_FILTER_FC} Hz, '
          f'fs={fs_slow:.2f} Hz')

# ---- Unwrap phase -> displacement ----
phase_raw = np.angle(iq_demod)
phase_unwrapped = np.unwrap(phase_raw)
displacement_mm = -phase_unwrapped * LAMBDA / (4 * np.pi) * 1000.0   # mm

# ---- Slow-time axis ----
N = len(iq_demod)
t = np.arange(N) / fs_slow

### 4.1 Displacement vs time

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(t, displacement_mm, linewidth=1.2)
plt.xlabel('Time (s)')
plt.ylabel('Displacement (mm)')
plt.title(f'Target displacement (from slow-time phase) — {scope_desc}, channel {CHANNEL}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

pp = np.ptp(displacement_mm)
print(f'Peak-to-peak displacement: {pp:.3f} mm  (expected ~1.6 mm)')

### 4.2 FFT of the I/Q signal (motion spectrum from the time-domain signal)

Spectrum of the complex slow-time phasor itself, before unwrapping. For a clean single moving target the motion appears as a peak near 0.2 Hz.

In [ ]:
N_pad = 4096
iq_centered = iq_demod - np.mean(iq_demod)
iq_fft = np.abs(np.fft.fft(iq_centered, N_pad))
freqs = np.fft.fftfreq(N_pad, d=1.0 / fs_slow)

# Show the positive-frequency half
pos = freqs >= 0
plt.figure(figsize=(10, 4))
plt.plot(freqs[pos], iq_fft[pos], linewidth=1.2)
plt.axvline(0.2, color='tab:green', linestyle='--', alpha=0.7, label='0.2 Hz')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.title(f'FFT of I/Q signal — {scope_desc}, channel {CHANNEL}')
plt.xlim([0, 2])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 FFT of the displacement waveform

Spectrum of the real displacement signal. Should peak at the plate's motion frequency (~0.2 Hz).

In [ ]:
disp_centered = displacement_mm - np.mean(displacement_mm)
disp_fft = np.abs(np.fft.fft(disp_centered, N_pad))
freqs_r = np.arange(N_pad) * fs_slow / N_pad
half = slice(0, N_pad // 2)

plt.figure(figsize=(10, 4))
plt.plot(freqs_r[half], disp_fft[half], linewidth=1.2)
plt.axvline(0.2, color='tab:green', linestyle='--', alpha=0.7, label='0.2 Hz')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.title(f'Displacement spectrum — {scope_desc}, channel {CHANNEL}')
plt.xlim([0, 2])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report the dominant motion frequency (skip DC)
peak_idx = np.argmax(disp_fft[1:N_pad // 2]) + 1
peak_freq = peak_idx * fs_slow / N_pad
print(f'Dominant motion frequency: {peak_freq:.3f} Hz  (expected ~0.2 Hz)')

## 5. Single-Chirp Selector + Range-FFT Sanity Check

Extract one fast-time chirp (the only place `chirp` is consumed) and confirm it sees the plate at 6 m. A complex FFT along fast-time turns the beat tone into a range peak. For RSET 1 (`RANGE_RES` = 7.82 cm/bin), a 6 m target should peak near bin `6 / 0.0782 ~ 77`.

This is the natural bridge into the later FPGA-RFFT comparison: it's the same operation, just on one chirp without windowing/averaging.

In [ ]:
# %%
# Single fast-time chirp for the selected channel: complex, length 128
chirp = ch_data[INSPECT_FRAME, :, INSPECT_CHIRP]
print(f'Selected chirp: frame {INSPECT_FRAME}, chirp {INSPECT_CHIRP}, '
      f'channel {CHANNEL} -> shape {chirp.shape}, dtype {chirp.dtype}')

# %%
# Complex range FFT of the single selected chirp (no window, raw)
rfft_chirp = np.fft.fft(chirp)
mag = np.abs(rfft_chirp)

n_bins = len(mag)
bins = np.arange(n_bins)
range_axis = bins * RANGE_RES   # meters per bin

# Expected target bin from geometry
expected_bin = 6.0 / RANGE_RES

# Locate the peak, skipping the DC/leakage region near bin 0
LEAKAGE_SKIP = 5
peak_bin = int(np.argmax(mag[LEAKAGE_SKIP:])) + LEAKAGE_SKIP
peak_range = peak_bin * RANGE_RES

print(f'Expected target bin (6 m):   ~{expected_bin:.1f}')
print(f'Measured peak bin:            {peak_bin}  ({peak_range:.2f} m)')

# %%
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle(f'Single-chirp range FFT — frame {INSPECT_FRAME}, '
             f'chirp {INSPECT_CHIRP}, channel {CHANNEL}')

# Left: magnitude vs range bin (full spectrum, including upper half)
axs[0].plot(bins, mag, linewidth=1.0)
axs[0].axvline(expected_bin, color='tab:green', linestyle='--',
               alpha=0.8, label=f'expected ~bin {expected_bin:.0f}')
axs[0].axvline(peak_bin, color='tab:red', linestyle=':',
               alpha=0.8, label=f'peak bin {peak_bin}')
axs[0].set_xlabel('Range bin')
axs[0].set_ylabel('|FFT|')
axs[0].set_title('Full spectrum (all 128 bins)')
axs[0].legend(); axs[0].grid(True, alpha=0.3)

# Right: magnitude vs range (lower half only, the unambiguous range region)
half_bins = n_bins // 2
axs[1].plot(range_axis[:half_bins], mag[:half_bins], linewidth=1.0)
axs[1].axvline(6.0, color='tab:green', linestyle='--', alpha=0.8, label='6 m')
axs[1].set_xlabel('Range (m)')
axs[1].set_ylabel('|FFT|')
axs[1].set_title('Lower half')
axs[1].legend(); axs[1].grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# RADC vs FPGA RFFT Comparison

Compare a range FFT computed from the RADC data against the FPGA's own RFFT output, taken from the **same joint recording** (a `.bin` containing interleaved `RADC` and `RFFT` frames).

Key idea: you **compute** the range FFT on RADC (the FPGA hasn't transformed it yet), but you **read** the RFFT directly (the FPGA already range-transformed it). Both reduce to a magnitude-vs-range-bin profile that should match in shape and peak location.

`get_frames` filters by header, so it separates the two frame types from the joint file automatically — call it once with `b'RADC'` and once with `b'RFFT'`.

### Joint-file path + RFFT decoder

In [ ]:
# Joint RADC+RFFT recording (interleaved frames). Kept separate from the
# RADC-only file used by sections 1–4.
RFFT_BINARY_FILEPATH = (
    "/home/shaylon/repos/ALiSM_Python_copy/vmd3_test/data/radc/2026-07-17/2026-07-17_16-24-12.bin"
)

# Optional manual bin (matches the earlier validated run). None = auto-detect.
CMP_MANUAL_TARGET_BIN = None


def decode_rfft_2d(frame):
    """
    Decode a 2D-mode RFFT payload (range-FFT'd, per chirp, complex).
    Datasheet layout: channel-major, then range bin, then sweep.
    Output shape: (128 range_bins, 64 chirps, 4 channels) to match RADC.
    """
    if len(frame) != 131072:
        raise ValueError(f'INVALID RFFT FRAME LENGTH FOR 2D MODE: {len(frame)}')
    raw = np.frombuffer(frame, dtype='<i2')          # 65536 int16
    raw = raw.reshape(4, 128, 64, 2)                 # (ch, range_bin, sweep, [Q,I])
    q_values = raw[:, :, :, 0].astype(np.float64)    # Q-first, matching RADC convention
    i_values = raw[:, :, :, 1].astype(np.float64)
    complex_data = i_values + 1j * q_values          # (ch, range_bin, sweep)
    return np.transpose(complex_data, (1, 2, 0))     # (range_bin, sweep, ch)

### Load both frame types and compute the two range profiles

`CMP_N_FRAMES` controls how many frames are averaged into each profile: 1 = single frame (mirrors the earlier validated run); >1 averages that many frames for a cleaner profile. `CMP_FRAME_START` is the first frame used (after the same `TRIM_FRAMES` startup skip).

In [ ]:
CMP_N_FRAMES    = 1   # how many frames to average into each profile (1 = single)
CMP_FRAME_START = TRIM_FRAMES   # first frame index to use (skip startup)
CMP_CHANNEL     = 'avg'  # 0 | 1 | 2 | 3 | 'avg'

# ---- Separate the two frame types from the joint file ----
radc_frames = get_frames(RFFT_BINARY_FILEPATH, b'RADC', CONFIG_MODE)
rfft_frames = get_frames(RFFT_BINARY_FILEPATH, b'RFFT', CONFIG_MODE)
print(f'Joint file: {len(radc_frames)} RADC frames, {len(rfft_frames)} RFFT frames')

n_avail = min(len(radc_frames), len(rfft_frames))
if n_avail == 0:
    raise RuntimeError('Need both RADC and RFFT frames to compare.')

# Clamp the requested window to what's available
start = CMP_FRAME_START
stop  = min(start + CMP_N_FRAMES, n_avail)
use_idx = range(start, stop)
print(f'Using frames [{start}:{stop}] ({stop - start} frame(s)) for the comparison.')


def _reduce_channels(mag_cube, channel):
    """
    Collapse a |.| cube (range_bins, chirps, channels) to a (range_bins,) profile.
    channel: int picks one RX channel (average over chirps only);
             'avg' averages over chirps AND channels.
    """
    if channel == 'avg':
        return np.mean(mag_cube, axis=(1, 2))        # over chirps + channels
    return np.mean(mag_cube[:, :, int(channel)], axis=1)  # one channel, over chirps


def range_profile_from_radc(frame, channel):
    """Compute |range FFT| profile from one RADC frame for the chosen channel(s)."""
    cube = decode_radc_2d(frame)                 # (samples, chirps, channels)
    my_rfft = np.fft.fft(cube, axis=0)           # range FFT along fast-time
    return _reduce_channels(np.abs(my_rfft), channel)


def range_profile_from_rfft(frame, channel):
    """Read |RFFT| profile from one FPGA RFFT frame for the chosen channel(s)."""
    fpga = decode_rfft_2d(frame)                 # (range_bins, sweeps, channels)
    return _reduce_channels(np.abs(fpga), channel)


# ---- Average the profiles over the chosen frame window ----
my_prof   = np.mean([range_profile_from_radc(radc_frames[i], CMP_CHANNEL) for i in use_idx], axis=0)
fpga_prof = np.mean([range_profile_from_rfft(rfft_frames[i], CMP_CHANNEL) for i in use_idx], axis=0)

ch_label = 'avg of all channels' if CMP_CHANNEL == 'avg' else f'channel {CMP_CHANNEL}'

nbins = my_prof.shape[0]
rng = np.arange(nbins) * MAX_RANGE / nbins   # range axis in meters

# ---- Peak bins (skip DC/leakage region) for a quick numeric check ----
SKIP = 10
my_peak   = int(np.argmax(my_prof[SKIP:])) + SKIP
fpga_peak = int(np.argmax(fpga_prof[SKIP:])) + SKIP
print(f'[{ch_label}] Computed-RFFT peak bin: {my_peak}  ({rng[my_peak]:.2f} m)')
print(f'[{ch_label}] FPGA-RFFT     peak bin: {fpga_peak}  ({rng[fpga_peak]:.2f} m)')
if CMP_MANUAL_TARGET_BIN is not None:
    print(f'(Expected target bin ~{CMP_MANUAL_TARGET_BIN}, '
          f'{rng[CMP_MANUAL_TARGET_BIN]:.2f} m)')

### Overlay: computed range FFT vs FPGA RFFT

Each profile is normalized to its own max so the **shapes** overlay directly regardless of absolute scaling. A matching peak location confirms the RADC decode and the RFFT decode agree.


In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(rng, my_prof / my_prof.max(), lw=1.4,
         label=f'Computed range FFT (RADC, {ch_label})')
plt.plot(rng, fpga_prof / fpga_prof.max(), lw=1.4, ls='--',
         label=f'FPGA RFFT ({ch_label})')
if CMP_MANUAL_TARGET_BIN is not None:
    plt.axvline(rng[CMP_MANUAL_TARGET_BIN], color='tab:green',
                linestyle=':', alpha=0.7,
                label=f'bin {CMP_MANUAL_TARGET_BIN} ({rng[CMP_MANUAL_TARGET_BIN]:.2f} m)')
plt.xlabel('Range (m)')
plt.ylabel('Normalized magnitude')
plt.title(f'RADC-computed range FFT vs FPGA RFFT — {ch_label} '
          f'({stop - start} frame(s) averaged)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()